# Retail Demand Forecasting & Inventory Optimization

## tl;dr

This notebook is the readable companion to the scripted pipeline. Run `python scripts/run_all.py` first to regenerate the outputs, then use this notebook to inspect the main data-quality, forecasting, and inventory-policy results.

## Context & Methods

The raw dataset is Kaggle's Store Item Demand Forecasting Challenge. The training file has one row per date, store, and item. The analysis compares a seasonal naive baseline with a gradient boosting model and then turns forecast uncertainty into safety stock and reorder points.

### Key Assumptions

- Observed sales are treated as demand because stockout flags are unavailable.
- Reorder logic assumes a 7-day lead time and 95% cycle service level.
- Dollar values use planning assumptions for price, gross margin, holding cost, and reorder cost.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'outputs').exists() and (ROOT.parent / 'outputs').exists():
    ROOT = ROOT.parent
OUTPUTS = ROOT / 'outputs'
FIGURES = ROOT / 'reports' / 'figures'
pd.set_option('display.max_columns', 30)

## Data

In [2]:
profile = pd.read_json(OUTPUTS / 'data_profile.json', typ='series')
profile

rows                        913000
columns                          4
start_date              2013-01-01
end_date                2017-12-31
stores                          10
items                           50
duplicate_grain_rows             0
missing_values                   0
negative_sales_rows              0
complete_daily_grid           True
dtype: object

In [3]:
display(Markdown((OUTPUTS / 'data_quality_report.md').read_text(encoding='utf-8')))

# Data Quality Report

## Dataset And Grain

The raw training file has **913,000 rows** and **4 columns**.
The intended grain is one row per calendar date, store, and item.

## Checks Performed

| check | evidence | status |
| --- | --- | --- |
| Expected grain | date x store x item | Pass |
| Duplicate grain rows | 0 | Pass |
| Missing values | 0 | Pass |
| Negative sales rows | 0 | Pass |
| Date coverage | 2013-01-01 to 2017-12-31 | Pass |
| Store/item coverage | 10 stores x 50 items | Pass |

## Findings

- The file is clean for the core forecasting task: no missing values, no duplicate
  date-store-item rows, no negative sales, and a complete daily grid.
- The biggest data-quality limitation is not a broken field. It is missing
  business context: the dataset has unit sales only, with no price, margin,
  on-hand inventory, stockout flags, promotions, supplier lead times, weather,
  holidays, or local events.
- Because sales are observed sales rather than true demand, a real stockout
  would censor demand. The Kaggle file does not tell us when that happened, so
  the model assumes historical sales are a good proxy for demand.

## Recommended Automated Tests

- Enforce uniqueness on `date`, `store`, and `item`.
- Reject negative sales and missing required fields.
- Check that each store-item series has continuous daily dates.
- Monitor row count by date if this became a recurring pipeline.


## Results

In [4]:
metrics = pd.read_csv(OUTPUTS / 'model_metrics.csv')
metrics

,model,rmse,mape,wmape,bias_units
0,gradient_boosting,7.773150,13.193579,10.979970,0.069458
1,moving_average_28,12.558698,21.065043,17.448038,2.696085
2,seasonal_naive,14.492021,22.815940,19.913708,-2.101543


In [5]:
costs = pd.read_csv(OUTPUTS / 'inventory_cost_summary.csv')
costs

,model,stockout_units,overstock_units,stockout_cost,holding_cost,avg_excess_inventory_value,reorder_cost,total_operating_cost
0,gradient_boosting,1603.966730,234617.291076,4491.106844,2924.681300,174.287131,245000.0,252415.788144
1,moving_average_28,1081.037755,475498.539218,3026.905713,5927.447544,353.227486,245000.0,253954.353257
2,seasonal_naive,3894.575706,320622.195177,10904.811978,3996.797228,238.176488,245000.0,259901.609205


In [6]:
sku = pd.read_csv(OUTPUTS / 'sku_difficulty.csv')
sku.head(10)

,store,item,mean_sales,sales_std,mape,rmse,cv,difficulty_score
0,7,5,13.326087,4.619854,30.645139,4.290939,0.346677,1.000
1,7,41,15.152174,5.186489,28.423010,4.259091,0.342293,0.996
2,7,27,15.043478,5.025021,28.539563,4.162004,0.334033,0.995
3,6,5,14.793478,4.843798,28.601540,4.335052,0.327428,0.994
4,7,4,15.967391,5.407466,24.939394,4.111231,0.338657,0.992
5,6,4,17.434783,5.713497,24.690982,4.599066,0.327707,0.989
6,5,41,17.195652,5.588920,27.053658,4.470605,0.325019,0.989
7,7,47,15.967391,5.037117,23.535833,3.875204,0.315463,0.983
8,5,27,17.228261,5.339266,27.257758,4.668912,0.309913,0.983
9,7,1,15.597826,4.932645,23.211377,4.120322,0.316239,0.981


## Takeaways

- The dataset is clean at the expected grain, but missing commercial context matters.
- The seasonal naive model is a fair baseline because demand has strong annual seasonality.
- The gradient boosting model improves validation accuracy and reduces simulated operating cost under the stated assumptions.
- The main operating recommendation is not one global inventory buffer; it is SKU-level safety stock where volatility and forecast error are highest.